In [34]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler,RobustScaler,StandardScaler
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,RobustScaler
import numpy as np
import pandas as pd
from scipy.stats import randint

In [35]:
train_df=pd.read_csv('cleaned_train.csv')
train_df.head()


,Inches,Ram,Weight,IS_Touchscreen,IPS,Retina Display,X_res,Y_res,PPI,Clock_Speed_GHz,...,OpSys_macOS,Processor_Type_Intel Core i3,Processor_Type_Intel Core i5,Processor_Type_Intel Core i7,Processor_Type_Other,Processor_Type_Other Intel Processor,Gpu_Brand_ARM,Gpu_Brand_Intel,Gpu_Brand_Nvidia,Price
0,0.0000,0.0,0.415584,0,0,0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,34035.2640
1,0.0000,-1.0,-0.181818,0,0,0,0.0,0.0,0.000000,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,29250.7200
2,-1.0000,-1.0,-0.519481,0,0,0,-554.0,-312.0,-0.975411,0.000000,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,41505.1200
3,-1.0000,-1.0,-0.571429,0,0,0,0.0,0.0,0.537685,-2.333333,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,17529.1200
4,-1.4375,0.0,-0.571429,1,1,0,0.0,0.0,0.813602,0.000000,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,42038.4528


In [36]:

X_train = train_df.drop(columns=['Price'])
y_train = train_df['Price']


In [37]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import numpy as np

models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),
    "Linear Regression":LinearRegression(),
    "XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

results = []

for name, model in models.items():
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_train)
    
    mae = mean_absolute_error(y_train, y_pred)
    rmse = np.sqrt(mean_squared_error(y_train, y_pred))
    r2 = r2_score(y_train, y_pred)
    
    results.append([name, mae, rmse, r2])

import pandas as pd

results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "R2"]
)

results_df

,Model,MAE,RMSE,R2
0,Decision Tree,385.256033,1911.634279,0.997139
1,Random Forest,4609.560390,7451.529329,0.956523
2,Gradient Boosting,10000.817151,13916.560122,0.848353
3,Linear Regression,13418.582750,19424.104053,0.704572
4,XGBoost,10177.008591,14213.507985,0.841812


In [38]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

gbr = GradientBoostingRegressor(random_state=42)

param_dist = {
    'n_estimators': [100, 150, 200, 300, 500],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.15],
    'max_depth': [2, 3, 4, 5, 6],
    'min_samples_split': [2, 4, 6, 10],
    'min_samples_leaf': [1, 2, 4, 6],
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=gbr,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)


,estimator,GradientBoost...ndom_state=42)
,param_distributions,"{'learning_rate': [0.01, 0.03, ...], 'max_depth': [2, 3, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 4, ...], ...}"
,n_iter,30
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [39]:

print("Best Parameters:")
print(random_search.best_params_)

print("Best CV R²:")
print(random_search.best_score_)

best_model = random_search.best_estimator_

y_pred = best_model.predict(X_train)

print("Test R²:", r2_score(y_train, y_pred))
print("Test MAE:", mean_absolute_error(y_train, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_train, y_pred)))

Best Parameters:
{'subsample': 0.7, 'n_estimators': 300, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_depth': 6, 'learning_rate': 0.03}
Best CV R²:
0.7230955509302789
Test R²: 0.9156448225968373
Test MAE: 6970.679196223762
Test RMSE: 10379.367477329177


In [41]:
results=[]
for name,model in models.items():
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )

    results.append([
        name,
        scores.mean(),
        scores.std()
    ])

In [42]:
import pandas as pd

cv_results = pd.DataFrame(
    results,
    columns=["Model", "Mean CV R2", "Std CV R2"]
)

cv_results.sort_values(
    "Mean CV R2",
    ascending=False
)

,Model,Mean CV R2,Std CV R2
1,Random Forest,0.697926,0.055612
4,XGBoost,0.694801,0.077426
2,Gradient Boosting,0.692333,0.084395
3,Linear Regression,0.652810,0.073640
0,Decision Tree,0.534957,0.071286


In [43]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

# 1. Base Model
rf = RandomForestRegressor(
    random_state=42
)

# 2. Parameters to search
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 5, 10, 15, 20, 25, 30],
    'min_samples_split': [2, 4, 6, 10],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 'log2', 0.5, 1.0]
}

# 3. Random Search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

# 4. Train using TRAIN data only
random_search.fit(X_train, y_train)

# 5. Best results
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV R²:")
print(random_search.best_score_)

Best Parameters:
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30}

Best CV R²:
0.7276954844426996


In [44]:
best_rf = random_search.best_estimator_

In [46]:
import joblib
joblib.dump(
    random_search.best_estimator_,
    "best_rf.pkl"
)
joblib.dump(
    best_model,
    "best_gbr.pkl"
)
joblib.dump(models["Decision Tree"], 
            "decision_tree.pkl")

joblib.dump(models["Gradient Boosting"], 
            "gradient_boosting.pkl")


joblib.dump(models["XGBoost"], 
            "XGBoost.pkl")

['XGBoost.pkl']